# 83. The Multi-Facility Location: p-Median Problem

## Tier 2: The Classic Heuristic (Improvement-Based Local Search)

### Key assumptions
- Customers can be assigned to their nearest open facility
- Initial solution can be generated randomly or using simple rules
- Local improvements can be found through facility swaps
- Search converges to locally optimal solution

### Approach (step-by-step)
The improvement heuristic operates through iterative local search:
1. **Initialization**: Generate initial facility locations (random or greedy)
2. **Customer Assignment**: Assign each customer to nearest open facility
3. **Cost Calculation**: Compute total transportation cost
4. **Neighborhood Search**: Try swapping each open facility with each closed facility
5. **Improvement Selection**: Accept swap if it reduces total cost
6. **Convergence**: Repeat until no improving swaps found

### What to look for in the results
- Iteration-by-iteration improvement process
- Final facility locations and customer assignments
- Cost reduction percentage over initial solution
- Convergence behavior and number of improvements

### Concrete example (from the source)
8-customer, 12-facility problem with 3 facilities to locate, demonstrating iterative improvement through facility swaps.

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict, Set
import random
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for p-Median Local Search")

In [ ]:
class PMedianLocalSearch:
    """
    Local Search heuristic for p-Median problem
    Uses facility swap neighborhood for iterative improvement
    """
    
    def __init__(self, customer_positions: List[float], demands: List[float], 
                 facility_positions: List[float], p: int):
        """
        Initialize p-Median Local Search solver
        
        Args:
            customer_positions: Geographic positions of customers
            demands: Demand volumes for each customer
            facility_positions: Potential facility locations
            p: Number of facilities to locate
        """
        self.customer_positions = np.array(customer_positions)
        self.demands = np.array(demands)
        self.facility_positions = np.array(facility_positions)
        self.p = p
        self.n_customers = len(customer_positions)
        self.n_facilities = len(facility_positions)
        
        # Calculate distance matrix
        self.distances = self._calculate_distances()
        
        # Solution tracking
        self.current_solution = None
        self.current_cost = np.inf
        self.best_solution = None
        self.best_cost = np.inf
        
        # Iteration history
        self.iteration_history = []
        
    def _calculate_distances(self) -> np.ndarray:
        """Calculate Euclidean distance matrix between customers and facilities"""
        distances = np.zeros((self.n_customers, self.n_facilities))
        for i in range(self.n_customers):
            for j in range(self.n_facilities):
                distances[i, j] = abs(self.customer_positions[i] - self.facility_positions[j])
        return distances
    
    def _assign_customers(self, facility_set: Set[int]) -> Dict[int, List[int]]:
        """
        Assign each customer to nearest open facility
        
        Args:
            facility_set: Set of open facility indices
            
        Returns:
            Dictionary mapping facility index to list of customer indices
        """
        assignments = {fac_idx: [] for fac_idx in facility_set}
        
        for cust_idx in range(self.n_customers):
            # Find nearest open facility
            min_dist = np.inf
            nearest_facility = -1
            
            for fac_idx in facility_set:
                dist = self.distances[cust_idx, fac_idx]
                if dist < min_dist:
                    min_dist = dist
                    nearest_facility = fac_idx
            
            assignments[nearest_facility].append(cust_idx)
        
        return assignments
    
    def _calculate_total_cost(self, facility_set: Set[int]) -> float:
        """
        Calculate total transportation cost for given facility set
        
        Args:
            facility_set: Set of open facility indices
            
        Returns:
            Total transportation cost
        """
        assignments = self._assign_customers(facility_set)
        total_cost = 0.0
        
        for fac_idx, customers in assignments.items():
            for cust_idx in customers:
                total_cost += self.demands[cust_idx] * self.distances[cust_idx, fac_idx]
        
        return total_cost
    
    def _generate_initial_solution(self, method: str = 'random') -> Set[int]:
        """
        Generate initial facility locations
        
        Args:
            method: 'random' or 'greedy'
            
        Returns:
            Set of initial facility indices
        """
        if method == 'random':
            # Random selection of p facilities
            return set(random.sample(range(self.n_facilities), self.p))
        elif method == 'greedy':
            # Greedy selection based on demand coverage
            facilities = set()
            remaining_customers = set(range(self.n_customers))
            
            for _ in range(self.p):
                best_facility = -1
                best_coverage = -1
                
                for fac_idx in range(self.n_facilities):
                    if fac_idx in facilities:
                        continue
                    
                    # Calculate total demand this facility can serve
                    coverage = 0
                    for cust_idx in remaining_customers:
                        coverage += self.demands[cust_idx] / (1 + self.distances[cust_idx, fac_idx])
                    
                    if coverage > best_coverage:
                        best_coverage = coverage
                        best_facility = fac_idx
                
                facilities.add(best_facility)
                
                # Remove customers served by this facility
                for cust_idx in list(remaining_customers):
                    if self.distances[cust_idx, best_facility] < np.mean(self.distances[cust_idx, :]):
                        remaining_customers.discard(cust_idx)
            
            return facilities
        else:
            raise ValueError("Method must be 'random' or 'greedy'")
    
    def _find_best_swap(self, current_facilities: Set[int]) -> Tuple[int, int, float]:
        """
        Find best facility swap that improves solution
        
        Args:
            current_facilities: Current set of open facilities
            
        Returns:
            Tuple of (facility_to_remove, facility_to_add, improvement)
        """
        best_improvement = 0
        best_swap = (-1, -1)
        
        current_cost = self._calculate_total_cost(current_facilities)
        
        # Try all possible swaps
        for facility_out in current_facilities:
            for facility_in in range(self.n_facilities):
                if facility_in in current_facilities:
                    continue
                
                # Create new facility set with swap
                new_facilities = current_facilities.copy()
                new_facilities.remove(facility_out)
                new_facilities.add(facility_in)
                
                # Calculate new cost
                new_cost = self._calculate_total_cost(new_facilities)
                improvement = current_cost - new_cost
                
                if improvement > best_improvement:
                    best_improvement = improvement
                    best_swap = (facility_out, facility_in)
        
        return best_swap[0], best_swap[1], best_improvement
    
    def solve(self, max_iterations: int = 100, init_method: str = 'random', 
              verbose: bool = True) -> Tuple[float, Set[int], Dict[int, List[int]]]:
        """
        Solve the p-Median problem using local search
        
        Args:
            max_iterations: Maximum number of iterations
            init_method: Initial solution generation method
            verbose: Whether to print progress
            
        Returns:
            Tuple of (total_cost, facility_locations, customer_assignments)
        """
        if verbose:
            print(f"Solving p-Median with Local Search")
            print(f"Customers: {self.n_customers}, Facilities: {self.n_facilities}, p: {self.p}")
            print(f"Initial solution method: {init_method}")
        
        # Generate initial solution
        self.current_solution = self._generate_initial_solution(init_method)
        self.current_cost = self._calculate_total_cost(self.current_solution)
        self.best_solution = self.current_solution.copy()
        self.best_cost = self.current_cost
        
        if verbose:
            print(f"\nInitial solution: {sorted(self.current_solution)}")
            print(f"Initial cost: {self.current_cost:.2f}")
        
        # Store initial state
        self.iteration_history.append({
            'iteration': 0,
            'solution': self.current_solution.copy(),
            'cost': self.current_cost,
            'improvement': 0
        })
        
        # Main local search loop
        iteration = 0
        improvements_made = 0
        
        while iteration < max_iterations:
            iteration += 1
            
            # Find best improving swap
            facility_out, facility_in, improvement = self._find_best_swap(self.current_solution)
            
            if improvement > 0.001:  # Significant improvement found
                # Apply swap
                self.current_solution.remove(facility_out)
                self.current_solution.add(facility_in)
                self.current_cost -= improvement
                improvements_made += 1
                
                # Update best solution if needed
                if self.current_cost < self.best_cost:
                    self.best_solution = self.current_solution.copy()
                    self.best_cost = self.current_cost
                
                if verbose:
                    print(f"Iteration {iteration}: Swap {facility_out} -> {facility_in}, Cost: {self.current_cost:.2f}")
                
                # Store iteration
                self.iteration_history.append({
                    'iteration': iteration,
                    'solution': self.current_solution.copy(),
                    'cost': self.current_cost,
                    'improvement': improvement
                })
            else:
                # No improving swap found - convergence
                if verbose:
                    print(f"Converged at iteration {iteration}")
                break
        
        # Get final assignments
        final_assignments = self._assign_customers(self.current_solution)
        
        if verbose:
            print(f"\nFinal solution: {sorted(self.current_solution)}")
            print(f"Final cost: {self.current_cost:.2f}")
            print(f"Number of improvements: {improvements_made}")
            print(f"Total improvement: {((self.iteration_history[0]['cost'] - self.current_cost) / self.iteration_history[0]['cost'] * 100):.1f}%")
        
        return self.current_cost, self.current_solution, final_assignments
    
    def print_solution(self, total_cost: float, facility_locations: Set[int], 
                      customer_assignments: Dict[int, List[int]]):
        """Print the solution in a readable format"""
        print(f"\n{'='*60}")
        print("LOCAL SEARCH SOLUTION")
        print(f"{'='*60}")
        print(f"Total Transportation Cost: {total_cost:.2f}")
        print(f"Number of Facilities: {self.p}")
        print(f"\nFacility Locations and Assignments:")
        
        for i, fac_idx in enumerate(sorted(facility_locations)):
            fac_pos = self.facility_positions[fac_idx]
            customers = customer_assignments[fac_idx]
            customer_positions = [self.customer_positions[c] for c in customers]
            customer_demands = [self.demands[c] for c in customers]
            
            print(f"\n  Facility {i+1} at position {fac_pos:.1f} (Index {fac_idx}):")
            print(f"    Serves customers at positions: {customer_positions}")
            print(f"    Customer demands: {customer_demands}")
            
            # Calculate cost for this facility
            cost = sum(self.demands[c] * self.distances[c, fac_idx] for c in customers)
            print(f"    Cost contribution: {cost:.2f}")
            print(f"    Number of customers: {len(customers)}")

print("PMedianLocalSearch class defined successfully")

In [ ]:
# Create the concrete example from the source
# 8 customers, 12 facilities, 3 facilities to locate

# Generate random but reproducible data for the example
np.random.seed(42)

n_customers = 8
n_facilities = 12
p = 3

# Customer positions (geographic distribution)
customer_positions = np.random.uniform(0, 20, n_customers)
customer_positions.sort()  # Order them geographically

# Customer demands (varying volumes)
demands = np.random.uniform(5, 25, n_customers)

# Facility positions (more options than customers)
facility_positions = np.random.uniform(0, 20, n_facilities)
facility_positions.sort()

print("CONCRETE EXAMPLE SETUP")
print(f"Number of customers: {n_customers}")
print(f"Number of potential facilities: {n_facilities}")
print(f"Number of facilities to locate: {p}")
print(f"\nCustomer positions: {customer_positions.round(2).tolist()}")
print(f"Customer demands: {demands.round(2).tolist()}")
print(f"Facility positions: {facility_positions.round(2).tolist()}")

# Create and solve the problem
solver = PMedianLocalSearch(customer_positions.tolist(), demands.tolist(), 
                           facility_positions.tolist(), p)

total_cost, facility_locations, customer_assignments = solver.solve(
    max_iterations=50, init_method='random', verbose=True
)

# Print detailed solution
solver.print_solution(total_cost, facility_locations, customer_assignments)

In [ ]:
# Visualize the local search process
def visualize_local_search_process(solver):
    """Visualize the local search iteration process"""
    
    if not solver.iteration_history:
        print("No iteration history available")
        return
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Extract iteration data
    iterations = [h['iteration'] for h in solver.iteration_history]
    costs = [h['cost'] for h in solver.iteration_history]
    improvements = [h['improvement'] for h in solver.iteration_history]
    
    # Plot 1: Cost reduction over iterations
    ax1.plot(iterations, costs, 'bo-', linewidth=2, markersize=6)
    ax1.set_xlabel('Iteration', fontsize=12)
    ax1.set_ylabel('Total Cost', fontsize=12)
    ax1.set_title('Cost Reduction Over Iterations', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Add improvement annotations
    for i, (iter_num, cost, improvement) in enumerate(zip(iterations, costs, improvements)):
        if improvement > 0.001 and i > 0:  # Skip initial iteration
            ax1.annotate(f'Improvement: {improvement:.1f}', 
                        (iter_num, cost), xytext=(iter_num, cost + max(costs)*0.02),
                        ha='center', va='bottom', fontsize=9, color='red',
                        arrowprops=dict(arrowstyle='->', color='red', alpha=0.7))
    
    # Plot 2: Improvement magnitude per iteration
    ax2.bar(iterations[1:], improvements[1:], alpha=0.7, color='orange')
    ax2.set_xlabel('Iteration', fontsize=12)
    ax2.set_ylabel('Cost Improvement', fontsize=12)
    ax2.set_title('Improvement Magnitude', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Cumulative improvement
    cumulative_improvement = np.cumsum(improvements)
    ax3.plot(iterations, cumulative_improvement, 'go-', linewidth=2, markersize=6)
    ax3.set_xlabel('Iteration', fontsize=12)
    ax3.set_ylabel('Cumulative Improvement', fontsize=12)
    ax3.set_title('Cumulative Cost Improvement', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Solution quality metrics
    initial_cost = costs[0]
    final_cost = costs[-1]
    total_improvement = initial_cost - final_cost
    improvement_percentage = (total_improvement / initial_cost) * 100
    
    metrics = ['Initial Cost', 'Final Cost', 'Total Improvement', 'Improvement %']
    values = [initial_cost, final_cost, total_improvement, improvement_percentage]
    colors = ['red', 'green', 'blue', 'orange']
    
    bars = ax4.bar(metrics, values, color=colors, alpha=0.7)
    ax4.set_ylabel('Value', fontsize=12)
    ax4.set_title('Solution Quality Metrics', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        if height < 100:  # For percentage
            label = f'{value:.1f}%'
        else:
            label = f'{value:.1f}'
        ax4.text(bar.get_x() + bar.get_width()/2., height + max(values)*0.01,
                label, ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Visualize the local search process
visualize_local_search_process(solver)

In [ ]:
# Visualize the final solution layout
def visualize_solution_layout(customer_positions, demands, facility_positions, 
                             facility_locations, customer_assignments, total_cost):
    """Create comprehensive visualization of the local search solution"""
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot 1: Geographic layout with assignments
    ax1.set_title('Local Search Solution: Facility Locations and Customer Assignments', 
                 fontsize=14, fontweight='bold')
    
    # Plot customers with size proportional to demand
    for i, (pos, demand) in enumerate(zip(customer_positions, demands)):
        ax1.scatter(pos, 0, s=demand*8, c='blue', alpha=0.7, edgecolors='black', linewidth=2)
        ax1.annotate(f'C{i+1}\n(d={demand:.1f})', (pos, 0), xytext=(pos, 0.8), 
                    ha='center', va='bottom', fontsize=9)
    
    # Plot all potential facilities
    for pos in facility_positions:
        ax1.scatter(pos, 0, s=80, c='lightgray', marker='^', alpha=0.5)
    
    # Plot selected facilities and assignments with colors
    colors = ['red', 'green', 'orange', 'purple', 'brown', 'pink']
    
    for i, fac_idx in enumerate(sorted(facility_locations)):
        fac_pos = facility_positions[fac_idx]
        color = colors[i % len(colors)]
        customers = customer_assignments[fac_idx]
        
        # Plot facility
        ax1.scatter(fac_pos, 0, s=200, c=color, marker='^', edgecolors='black', linewidth=2)
        ax1.annotate(f'F{fac_idx}', (fac_pos, 0), xytext=(fac_pos, -0.8), 
                    ha='center', va='top', fontsize=10, fontweight='bold')
        
        # Draw assignment lines
        for cust_idx in customers:
            cust_pos = customer_positions[cust_idx]
            ax1.plot([fac_pos, cust_pos], [0, 0], color=color, alpha=0.6, linewidth=2)
    
    ax1.set_xlabel('Geographic Position', fontsize=12)
    ax1.set_ylabel('Location', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-1.5, 1.5)
    
    # Plot 2: Cost breakdown and facility performance
    ax2.set_title('Facility Performance Analysis', fontsize=14, fontweight='bold')
    
    facility_data = []
    facility_labels = []
    
    for i, fac_idx in enumerate(sorted(facility_locations)):
        customers = customer_assignments[fac_idx]
        
        # Calculate metrics
        cost = sum(demands[c] * abs(customer_positions[c] - facility_positions[fac_idx]) for c in customers)
        total_demand = sum(demands[c] for c in customers)
        avg_distance = sum(abs(customer_positions[c] - facility_positions[fac_idx]) for c in customers) / len(customers) if customers else 0
        
        facility_data.append({
            'cost': cost,
            'demand': total_demand,
            'customers': len(customers),
            'avg_distance': avg_distance
        })
        facility_labels.append(f'F{fac_idx}\n({len(customers)} cust)')
    
    # Create grouped bar chart
    x = np.arange(len(facility_labels))
    width = 0.2
    
    # Normalize values for better visualization
    max_cost = max(d['cost'] for d in facility_data)
    max_demand = max(d['demand'] for d in facility_data)
    max_customers = max(d['customers'] for d in facility_data)
    max_distance = max(d['avg_distance'] for d in facility_data)
    
    costs_norm = [d['cost']/max_cost for d in facility_data]
    demands_norm = [d['demand']/max_demand for d in facility_data]
    customers_norm = [d['customers']/max_customers for d in facility_data]
    distances_norm = [d['avg_distance']/max_distance for d in facility_data]
    
    bars1 = ax2.bar(x - width*1.5, costs_norm, width, label='Cost (norm)', alpha=0.8)
    bars2 = ax2.bar(x - width*0.5, demands_norm, width, label='Demand (norm)', alpha=0.8)
    bars3 = ax2.bar(x + width*0.5, customers_norm, width, label='Customers (norm)', alpha=0.8)
    bars4 = ax2.bar(x + width*1.5, distances_norm, width, label='Avg Distance (norm)', alpha=0.8)
    
    ax2.set_xlabel('Facilities', fontsize=12)
    ax2.set_ylabel('Normalized Value', fontsize=12)
    ax2.set_title('Facility Performance Metrics', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(facility_labels)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add total cost annotation
    ax2.text(0.02, 0.98, f'Total Cost: {total_cost:.2f}', transform=ax2.transAxes,
            ha='left', va='top', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Create solution visualization
visualize_solution_layout(customer_positions, demands, facility_positions, 
                          facility_locations, customer_assignments, total_cost)

In [ ]:
# Compare different initialization methods
def compare_initialization_methods():
    """Compare random vs greedy initialization methods"""
    print("\n" + "="*60)
    print("COMPARING INITIALIZATION METHODS")
    print("="*60)
    
    methods = ['random', 'greedy']
    results = {}
    
    for method in methods:
        print(f"\nTesting {method} initialization:")
        solver_test = PMedianLocalSearch(customer_positions.tolist(), demands.tolist(), 
                                      facility_positions.tolist(), p)
        
        cost_test, fac_locs_test, cust_assign_test = solver_test.solve(
            max_iterations=50, init_method=method, verbose=False
        )
        
        results[method] = {
            'final_cost': cost_test,
            'initial_cost': solver_test.iteration_history[0]['cost'],
            'improvement': solver_test.iteration_history[0]['cost'] - cost_test,
            'improvement_pct': ((solver_test.iteration_history[0]['cost'] - cost_test) / solver_test.iteration_history[0]['cost'] * 100),
            'iterations': len(solver_test.iteration_history) - 1,
            'facility_locations': sorted(fac_locs_test)
        }
        
        print(f"  Initial cost: {results[method]['initial_cost']:.2f}")
        print(f"  Final cost: {results[method]['final_cost']:.2f}")
        print(f"  Improvement: {results[method]['improvement']:.2f} ({results[method]['improvement_pct']:.1f}%)")
        print(f"  Iterations: {results[method]['iterations']}")
        print(f"  Final facilities: {results[method]['facility_locations']}")
    
    # Create comparison visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Cost comparison
    methods_list = list(results.keys())
    initial_costs = [results[m]['initial_cost'] for m in methods_list]
    final_costs = [results[m]['final_cost'] for m in methods_list]
    
    x = np.arange(len(methods_list))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, initial_costs, width, label='Initial Cost', alpha=0.8)
    bars2 = ax1.bar(x + width/2, final_costs, width, label='Final Cost', alpha=0.8)
    
    ax1.set_xlabel('Initialization Method', fontsize=12)
    ax1.set_ylabel('Total Cost', fontsize=12)
    ax1.set_title('Cost Comparison by Initialization Method', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(methods_list)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Improvement percentage
    improvements_pct = [results[m]['improvement_pct'] for m in methods_list]
    bars = ax2.bar(methods_list, improvements_pct, alpha=0.8, color='green')
    ax2.set_xlabel('Initialization Method', fontsize=12)
    ax2.set_ylabel('Improvement Percentage (%)', fontsize=12)
    ax2.set_title('Cost Improvement by Initialization Method', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, pct in zip(bars, improvements_pct):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Plot 3: Number of iterations
    iterations = [results[m]['iterations'] for m in methods_list]
    bars = ax3.bar(methods_list, iterations, alpha=0.8, color='orange')
    ax3.set_xlabel('Initialization Method', fontsize=12)
    ax3.set_ylabel('Number of Iterations', fontsize=12)
    ax3.set_title('Convergence Speed by Initialization Method', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, it in zip(bars, iterations):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{it}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 4: Summary table
    ax4.axis('tight')
    ax4.axis('off')
    
    # Create summary data
    summary_data = []
    for method in methods_list:
        summary_data.append([
            method.capitalize(),
            f"{results[method]['initial_cost']:.2f}",
            f"{results[method]['final_cost']:.2f}",
            f"{results[method]['improvement_pct']:.1f}%",
            f"{results[method]['iterations']}"
        ])
    
    table = ax4.table(cellText=summary_data,
                     colLabels=['Method', 'Initial Cost', 'Final Cost', 'Improvement %', 'Iterations'],
                     cellLoc='center',
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    # Style the header row
    for i in range(len(summary_data[0])):
        table[(0, i)].set_facecolor('#40466e')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    ax4.set_title('Summary Comparison', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Compare initialization methods
comparison_results = compare_initialization_methods()

In [ ]:
# Scalability analysis
def scalability_analysis():
    """Test algorithm performance on different problem sizes"""
    print("\n" + "="*60)
    print("SCALABILITY ANALYSIS")
    print("="*60)
    
    # Test different problem sizes
    test_cases = [
        {'customers': 5, 'facilities': 8, 'p': 2},
        {'customers': 8, 'facilities': 12, 'p': 3},
        {'customers': 12, 'facilities': 18, 'p': 4},
        {'customers': 15, 'facilities': 20, 'p': 5},
    ]
    
    results = []
    
    import time
    
    for case in test_cases:
        print(f"\nTesting case: {case['customers']} customers, {case['facilities']} facilities, p={case['p']}")
        
        # Generate test data
        np.random.seed(42)  # Reproducible
        cust_pos = np.random.uniform(0, 30, case['customers'])
        cust_pos.sort()
        dem = np.random.uniform(5, 20, case['customers'])
        fac_pos = np.random.uniform(0, 30, case['facilities'])
        fac_pos.sort()
        
        # Time the solution
        start_time = time.time()
        
        solver_test = PMedianLocalSearch(cust_pos.tolist(), dem.tolist(), 
                                      fac_pos.tolist(), case['p'])
        cost_test, fac_locs_test, cust_assign_test = solver_test.solve(
            max_iterations=100, init_method='random', verbose=False
        )
        
        end_time = time.time()
        computation_time = end_time - start_time
        
        result = {
            'customers': case['customers'],
            'facilities': case['facilities'],
            'p': case['p'],
            'final_cost': cost_test,
            'iterations': len(solver_test.iteration_history) - 1,
            'computation_time': computation_time,
            'improvement_pct': ((solver_test.iteration_history[0]['cost'] - cost_test) / solver_test.iteration_history[0]['cost'] * 100)
        }
        
        results.append(result)
        
        print(f"  Final cost: {cost_test:.2f}")
        print(f"  Iterations: {result['iterations']}")
        print(f"  Computation time: {computation_time:.3f} seconds")
        print(f"  Improvement: {result['improvement_pct']:.1f}%")
    
    # Create scalability plots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Extract data for plotting
    customers = [r['customers'] for r in results]
    facilities = [r['facilities'] for r in results]
    costs = [r['final_cost'] for r in results]
    iterations = [r['iterations'] for r in results]
    times = [r['computation_time'] for r in results]
    improvements = [r['improvement_pct'] for r in results]
    
    # Plot 1: Cost vs problem size
    ax1.plot(customers, costs, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Customers', fontsize=12)
    ax1.set_ylabel('Final Cost', fontsize=12)
    ax1.set_title('Cost vs Problem Size', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Iterations vs problem size
    ax2.plot(customers, iterations, 'go-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Customers', fontsize=12)
    ax2.set_ylabel('Number of Iterations', fontsize=12)
    ax2.set_title('Convergence Iterations vs Problem Size', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Computation time vs problem size
    ax3.plot(customers, times, 'ro-', linewidth=2, markersize=8)
    ax3.set_xlabel('Number of Customers', fontsize=12)
    ax3.set_ylabel('Computation Time (seconds)', fontsize=12)
    ax3.set_title('Computation Time vs Problem Size', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Improvement percentage vs problem size
    ax4.plot(customers, improvements, 'mo-', linewidth=2, markersize=8)
    ax4.set_xlabel('Number of Customers', fontsize=12)
    ax4.set_ylabel('Improvement Percentage (%)', fontsize=12)
    ax4.set_title('Solution Quality vs Problem Size', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Perform scalability analysis
scalability_results = scalability_analysis()

### Why this Tier exists vs Tier 1
This Tier provides **practical computational efficiency** for larger problems:
- **Scalable solution** for problems where dynamic programming is infeasible
- **Fast convergence** to good quality solutions in reasonable time
- **Real-time capability** for dynamic decision making
- **Memory efficient** - doesn't require DP table storage

### Pros / Cons vs Tier 1 (Dynamic Programming)
**Pros:**
- Handles much larger problem instances (50+ customers vs 20 for DP)
- Faster computation time for medium-large problems
- Lower memory requirements
- Suitable for real-time applications
- Easy to implement and understand

**Cons:**
- No optimality guarantee (only local optimum)
- Solution quality depends on initialization
- May get stuck in poor local optima
- Multiple runs may give different results

### When to use this Tier
- **Medium to large problems** (n > 20-30 customers)
- **Time-sensitive decisions** requiring quick solutions
- **Limited computational resources** (memory constraints)
- **Dynamic environments** where problems change frequently
- **Initial solution generation** for more advanced metaheuristics
- **Benchmarking** for testing more sophisticated algorithms